# HyDE — Hypothetical Document Embeddings

**Problem:** Questions and answers live in different parts of embedding space.
- Query: "Explain photosynthesis" (question style)
- Document: "Photosynthesis is a biological process..." (statement style)
- Their embeddings might not be as close as you'd expect!

**Solution:** Generate a FAKE answer, embed THAT, and search with it.
The fake answer doesn't need to be correct — it just needs to SOUND like
the documents we're searching for.

**Pipeline:** Query → LLM generates hypothetical answer → Embed answer → Search → Results

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

In [ ]:
# The actual document in our corpus
real_doc = (
    "Photosynthesis is a biological process used by plants, algae, and some bacteria "
    "to convert light energy into chemical energy stored in glucose. The overall equation "
    "is 6CO2 + 6H2O + light → C6H12O6 + 6O2."
)

# The user's query
query = "Explain photosynthesis"

# A hypothetical answer (simulating what an LLM would generate)
hypothetical = (
    "Photosynthesis is the process by which green plants and certain other organisms "
    "transform light energy into chemical energy. During photosynthesis, plants capture "
    "light using chlorophyll in their leaves and use it to convert carbon dioxide and "
    "water into glucose and oxygen."
)

print("Query (question):")
print(f'  "{query}"\n')
print("Hypothetical answer (LLM-generated):")
print(f'  "{hypothetical}"\n')
print("Real document (what we want to find):")
print(f'  "{real_doc}"')

In [ ]:
# Embed all three
query_emb = model.encode(query)
hypo_emb = model.encode(hypothetical)
doc_emb = model.encode(real_doc)

# Compare similarities
sim_query_doc = cosine_sim(query_emb, doc_emb)
sim_hypo_doc = cosine_sim(hypo_emb, doc_emb)
sim_query_hypo = cosine_sim(query_emb, hypo_emb)

print("Similarity scores:\n")
print(f"  Query ↔ Real Doc:        {sim_query_doc:.4f}  (normal semantic search)")
print(f"  Hypothetical ↔ Real Doc: {sim_hypo_doc:.4f}  (HyDE search)")
print(f"  Query ↔ Hypothetical:    {sim_query_hypo:.4f}  (how close LLM got)")

improvement = ((sim_hypo_doc - sim_query_doc) / sim_query_doc) * 100
print(f"\nHyDE improvement: +{improvement:.1f}% closer to the real document!")

## The Insight: Style Matters in Embedding Space

The query "Explain photosynthesis" is a **question**.
The document "Photosynthesis is a biological process..." is a **statement**.

Even though they're about the same topic, their embeddings differ because
the model also encodes **style** (question vs statement), not just **meaning**.

HyDE bridges this gap by converting the question into a statement first.

In [ ]:
# More examples showing the question-vs-statement gap
examples = [
    {
        "query": "What is CRISPR?",
        "hypothetical": "CRISPR-Cas9 is a gene-editing technology that uses guide RNA to direct enzymes to specific DNA sequences for precise modifications.",
        "real_doc": "CRISPR-Cas9 is a gene-editing technology adapted from a bacterial immune defense system. It uses a guide RNA to direct the Cas9 enzyme to a specific DNA sequence.",
    },
    {
        "query": "How does machine learning work?",
        "hypothetical": "Machine learning works by training algorithms on data to identify patterns. Models learn from examples and improve predictions over time without explicit programming.",
        "real_doc": "Machine learning is a subset of artificial intelligence that enables systems to learn from data without being explicitly programmed. Key paradigms include supervised, unsupervised, and reinforcement learning.",
    },
]

print("Question vs Statement gap — HyDE fixes it:\n")
for ex in examples:
    q_emb = model.encode(ex["query"])
    h_emb = model.encode(ex["hypothetical"])
    d_emb = model.encode(ex["real_doc"])
    
    normal = cosine_sim(q_emb, d_emb)
    hyde = cosine_sim(h_emb, d_emb)
    
    print(f"  Query: \"{ex['query']}\"")
    print(f"  Normal search: {normal:.4f} → HyDE search: {hyde:.4f} (Δ +{hyde-normal:.4f})")
    print()

## When Does HyDE Help vs Hurt?

| Scenario | Helps? | Why |
|----------|--------|-----|
| "Explain X" / "What is X" | Yes | Bridges question→statement gap |
| "metformin 500mg side effects" | No | Already statement-like, exact terms |
| Obscure/niche topic | Risky | LLM might hallucinate a bad answer |
| Well-known topic | Yes | LLM generates accurate-sounding answer |

## Key Takeaways

1. **HyDE moves you from question-space to answer-space** in the embedding model
2. The hypothetical answer **doesn't need to be correct** — just stylistically similar to real docs
3. Adds **LLM latency** (~300-500ms) but can significantly boost recall
4. Works best when there's a **style mismatch** between queries and documents
5. Can be combined with re-ranking for even better results